# FractalDSL — Notebook da Entrega Parcial 2

Este notebook contém a implementação completa da FractalDSL como uma DSL embutida em **Guile/Scheme** e os exemplos selecionados no README.

A estrutura do notebook acompanha a ordem em que as primitivas aparecem no tutorial:

1. Estrutura de dados do fractal (`create-fractal`, `set-field`, `get-field`).
2. Aritmética complexa.
3. Primitivas de parâmetro (`equation`, `constant`, `iterations`, `center`, `zoom`, `resolution`, `color`).
4. Sistemas de Funções Iteradas (`ifs`, `transform`, `affine`) e composição (`with-depth`).
5. Parser mínimo de equações e ponto de entrada `generate`.
6. Os quatro exemplos do README.

## 1. Estrutura do fractal

Um fractal é uma *association list* de campos. `create-fractal` devolve a estrutura base; `set-field` e `get-field` atualizam e consultam campos sem mutação.

In [ ]:
(define (create-fractal name)
  (list
    `(name       . ,name)
    '(equation   . #nil)
    '(constants  . ())
    '(iterations . 0)
    '(center     . (0 0))
    '(zoom       . 100)))

(define (set-field fractal field value)
  (map (lambda (sublist)
         (if (eq? (car sublist) field)
             (cons field value)
             sublist))
       fractal))

(define (get-field fractal field)
  (let ((result (assq field fractal)))
    (if result (cdr result) #nil)))

## 2. Aritmética complexa

Um complexo é representado pelo par `(cons real imag)`. Estas operações são a base para a avaliação de qualquer equação iterativa.

In [ ]:
(define (make-c r i) (cons r i))
(define (c-real z) (car z))
(define (c-imag z) (cdr z))

(define (c+ a b)
  (make-c (+ (c-real a) (c-real b)) (+ (c-imag a) (c-imag b))))

(define (c- a b)
  (make-c (- (c-real a) (c-real b)) (- (c-imag a) (c-imag b))))

(define (c* a b)
  (make-c (- (* (c-real a) (c-real b)) (* (c-imag a) (c-imag b)))
          (+ (* (c-real a) (c-imag b)) (* (c-imag a) (c-real b)))))

(define (c-abs z)
  (sqrt (+ (* (c-real z) (c-real z)) (* (c-imag z) (c-imag z)))))

(define (c-pow z n)
  (if (= n 0) (make-c 1 0) (c* z (c-pow z (- n 1)))))

## 3. Primitivas de parâmetro

Cada primitiva recebe um fractal como primeiro argumento e devolve um novo fractal com o campo correspondente atualizado. Essa convenção permite encadear chamadas formando um pipeline funcional.

In [ ]:
(define (equation fractal expr-str)
  (set-field fractal 'equation expr-str))

(define (constant fractal name value)
  (let ((cs (get-field fractal 'constants)))
    (set-field fractal 'constants
      (cons (cons name value)
            (if (eq? cs #nil) '() cs)))))

(define (iterations fractal n)
  (set-field fractal 'iterations n))

(define (center fractal re im)
  (set-field fractal 'center (list re im)))

(define (zoom fractal z)
  (set-field fractal 'zoom z))

(define (resolution fractal w h)
  (set-field fractal 'resolution (list w h)))

(define (color fractal scheme)
  (set-field fractal 'color scheme))

## 4. Sistemas de Funções Iteradas e composição

`ifs` é uma macro que anexa ao fractal uma lista de `transform`s. Cada `transform` associa uma probabilidade a uma transformação afim — ou, quando combinada com `with-depth`, a outro fractal já definido.

In [ ]:
(define (affine a b c d e f)
  (list 'affine a b c d e f))

(define (transform prob aff)
  (list 'transform prob aff))

(define-macro (ifs fractal . transforms)
  `(cons (cons 'ifs (list ,@transforms)) ,fractal))

(define-macro (with-depth d fractal)
  `(cons (cons 'depth ,d) ,fractal))

(define (fractal-with-depth? x)
  (and (list? x) (assq 'depth x)))

(define (apply-affine aff point)
  (let ((a (list-ref aff 1)) (b (list-ref aff 2))
        (c (list-ref aff 3)) (d (list-ref aff 4))
        (e (list-ref aff 5)) (f (list-ref aff 6))
        (x (car point))     (y (cadr point)))
    (list (+ (* a x) (* b y) e)
          (+ (* c x) (* d y) f))))

(define (choose-transform transforms)
  (let ((r (random:uniform)))
    (let loop ((ts transforms) (acc 0))
      (let* ((t       (car ts))
             (prob    (cadr t))
             (new-acc (+ acc prob)))
        (if (or (< r new-acc) (null? (cdr ts)))
            t
            (loop (cdr ts) new-acc))))))

(define (iterate-ifs fractal n)
  (let ((transforms (get-field fractal 'ifs)))
    (let loop ((point '(0.0 0.0)) (points '()) (i 0))
      (if (= i n)
          (reverse points)
          (let* ((t   (choose-transform transforms))
                 (val (caddr t)))
            (cond
              ((fractal-with-depth? val)
               (let* ((d       (get-field val 'depth))
                      (sub-pts (iterate-ifs val d))
                      (new-point (if (null? sub-pts) point
                                     (car (reverse sub-pts)))))
                 (loop new-point (append points sub-pts) (+ i 1))))
              (else
               (let ((new-point (apply-affine val point)))
                 (loop new-point (cons new-point points) (+ i 1))))))))))

## 5. Execução: parser de equação + `generate`

O parser atual reconhece `z^n ± c` e `z*z+c`. `generate` inspeciona o fractal e escolhe o mecanismo de iteração: `ifs` se presente, caso contrário a equação.

In [ ]:
(define (parse-exponent s)
  (let ((hat (string-contains s "^")))
    (and hat
         (let* ((rest  (substring s (+ hat 1)))
                (plus  (string-contains rest "+"))
                (minus (string-contains rest "-"))
                (end   (or plus minus (string-length rest))))
           (string->number (substring rest 0 end))))))

(define (parse-expr str)
  (let ((s (string-trim str)))
    (cond
      ((and (string-contains s "z^") (string-contains s "+c"))
       (let ((n (parse-exponent s)))
         (lambda (z c) (c+ (c-pow z n) c))))
      ((and (string-contains s "z^") (string-contains s "-c"))
       (let ((n (parse-exponent s)))
         (lambda (z c) (c- (c-pow z n) c))))
      ((string-contains s "z^")
       (let ((n (parse-exponent s)))
         (lambda (z c) (c-pow z n))))
      ((string=? s "z*z+c")
       (lambda (z c) (c+ (c* z z) c)))
      (else (error "Equação não reconhecida" str)))))

(define (parse-equation str)
  (let* ((sides (string-split str #\=))
         (rhs   (string-trim (cadr sides))))
    (parse-expr rhs)))

(define (iterate-equation f max-iter c)
  (let loop ((z (make-c 0.0 0.0)) (i 0))
    (cond ((= i max-iter) i)
          ((> (c-abs z) 2.0) i)
          (else (loop (f z c) (+ i 1))))))

(define (generate fractal)
  (let ((eq-str  (get-field fractal 'equation))
        (ifs-val (get-field fractal 'ifs))
        (iters   (get-field fractal 'iterations)))
    (cond
      ((not (eq? ifs-val #nil))
       (iterate-ifs fractal iters))
      ((string? eq-str)
       (let ((f (parse-equation eq-str)))
         (iterate-equation f iters (make-c 0.0 0.0))))
      (else
       (error "Fractal sem equation nem ifs" (get-field fractal 'name))))))

## 6. Exemplos Selecionados

Os mesmos quatro exemplos listados no README.

### 6.1 Mandelbrot — equação como configuração

In [ ]:
(define mandelbrot
  (zoom
    (center
      (iterations
        (equation (create-fractal "Mandelbrot") "z=z^2+c")
        500)
      -0.5 0)
    200))

(display "Mandelbrot, c=(0,0): ")
(display (generate mandelbrot))
(newline)

### 6.2 Julia — parametrização com `constant`

In [ ]:
(define julia
  (zoom
    (center
      (constant
        (iterations
          (equation (create-fractal "Julia") "z=z^2+c")
          500)
        'c (cons -0.4 0.6))
      0 0)
    150))

(display "Julia — constants: ")
(display (get-field julia 'constants))
(newline)

### 6.3 Sierpinski — IFS e macros

In [ ]:
(define sierpinski
  (iterations
    (ifs (create-fractal "Sierpinski")
      (transform 0.33 (affine 0.5 0.0 0.0 0.5 0.0  0.0))
      (transform 0.33 (affine 0.5 0.0 0.0 0.5 0.5  0.0))
      (transform 0.34 (affine 0.5 0.0 0.0 0.5 0.25 0.5)))
    8))

(display "Sierpinski — primeiros 8 pontos:") (newline)
(for-each (lambda (p) (display p) (newline))
          (generate sierpinski))

### 6.4 Combinado — `with-depth` e fractais como valores

Um fractal cujos ramos são outros fractais já definidos, cada um expandido até uma profundidade própria.

In [ ]:
(define barnsley
  (ifs (create-fractal "BarnsleyFern")
    (transform 0.01 (affine  0.00  0.00  0.00  0.16  0.00  0.00))
    (transform 0.85 (affine  0.85  0.04 -0.04  0.85  0.00  1.60))
    (transform 0.07 (affine  0.20 -0.26  0.23  0.22  0.00  1.60))
    (transform 0.07 (affine -0.15  0.28  0.26  0.24  0.00  0.44))))

(define sierpinski-base
  (ifs (create-fractal "Sierpinski")
    (transform 0.33 (affine 0.5 0.0 0.0 0.5 0.0  0.0))
    (transform 0.33 (affine 0.5 0.0 0.0 0.5 0.5  0.0))
    (transform 0.34 (affine 0.5 0.0 0.0 0.5 0.25 0.5))))

(define combinado
  (iterations
    (ifs (create-fractal "Combinado")
      (transform 0.5 (with-depth 5 sierpinski-base))
      (transform 0.5 (with-depth 3 barnsley)))
    6))

(display "Combinado — amostra de pontos:") (newline)
(for-each (lambda (p) (display p) (newline))
          (generate combinado))